In [1]:
#!/usr/bin/env python
# coding: utf-8
#
import numpy as np
import pandas as pd
import random
import glob
import time
import pickle
import pywt

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
#
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
#!pip install pandas==1.5.3
#!pip install tensorflow==2.15.0

In [3]:
pd.__version__

'1.5.3'

In [4]:
tf.__version__

'2.15.0'

In [6]:
#
#
## Model constants
BATCH_SIZE = 32
learning_rate = 1e-3
EPOCHS = 400
#
# 1D input data
#out_dim = 1200  # nondec. DWT
#out_dim = 224  # dec. DWT
#out_dim = 113  # dec. clip. DWT
out_dim = 200   # orig. signal
drop_rate = 0.3
#
## Data and training constants
#refDir = '/home/ec2-user/SageMaker/ref/'
refDir = 'C:/Users/kocha2/Documents/Projects/InnovationDays2024/Data/FSpdf/TrainTestPathsNew/'
#resDir = 'C:/Users/kocha2/Documents/Projects/InnovationDays2024/Data/Test_Results/DWT_DNN_New/'
resDir = 'C:/Users/kocha2/Documents/Projects/InnovationDays2024/Data/Test_Results/PMF_DNN_New/'
fResName = 'DecDWT_Dense_TestPerf_df_N4P5_split'
#
Nsplits = 20
Ntr_inst = 20
lossThresh = 0.1
rangeThresh = 0.15
#
#
##  Functions
#
def load_data(fPaths):
    FSpdfs = []
    for fPath in fPaths:
        with open(fPath, 'rb') as handle:
            FSpdf = pickle.load(handle)
        FSpdfs.append(FSpdf)
    #
    return np.stack(FSpdfs, axis=0)
#
# function for creating datasets of decimated DWT coeffs concatenated in 1D vector
def dec_dwt1D(X_signal, wavelet='db5', n_max=3):
    X = []
    for i in range(X_signal.shape[0]):
        signal = X_signal[i, :]
        coeff1D = np.array([])
        coeff_list = pywt.wavedec(signal, wavelet=wavelet, level=n_max)
        for coeff in coeff_list:
            coeff1D = np.concatenate((coeff1D, coeff))
        X.append(coeff1D)
    X = np.stack(X, axis=0)
    #
    return X
#
# split X_test in neg and pos parts
def split_Test_neg_pos(X_test, Y_test):
    X_test_neg = X_test[~np.bool_(Y_test)]
    X_test_neg = X_test_neg.reshape(X_test_neg.shape[0], 1, X_test_neg.shape[1])  # 1D case, Dense
    #
    X_test_pos = X_test[np.bool_(Y_test)]
    X_test_pos = X_test_pos.reshape(X_test_pos.shape[0], 1, X_test_pos.shape[1])  # 1D case, Dense
    #
    Y_test_neg = np.zeros(X_test_neg.shape[0])
    Y_test_pos = np.ones(X_test_pos.shape[0])
    #
    return X_test_neg, X_test_pos, Y_test_neg, Y_test_pos
#
# function to define DNN model
def DNN_model(out_dim, drop_rate, learning_rate):
    # define DNN
    model = tf.keras.Sequential()
    model.add(layers.Dense(64, activation='leaky_relu', input_shape=[1, out_dim]))
    model.add(layers.Dropout(drop_rate))
    #
    model.add(layers.Dense(32, activation='leaky_relu'))
    model.add(layers.Dropout(drop_rate))
    #
    model.add(layers.Dense(16, activation='leaky_relu'))
    model.add(layers.Dropout(drop_rate))
    #
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    #
    cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate)
    #
    return (model, cross_entropy, optimizer)
#
# function to create test dfs data for sample IDs to estimate test performance in terms of original samples
def make_test_dfs(fpathsTst, labelsTst):
    #
    sid_heal_df = pd.DataFrame(np.empty((len(labelsTst)-sum(labelsTst), 5)), columns=['FCid', 'Sid', 'Subid', 'ExSid', 'Pred'])
    sid_canc_df = pd.DataFrame(np.empty((sum(labelsTst), 5)), columns=['FCid', 'Sid', 'Subid', 'ExSid', 'Pred'])
    #
    i_heal = 0
    i_canc = 0
    for fPath in fpathsTst:
        path = fPath.split('_')
        cohort = path[0].split('/')[-1]
        FCid = path[1].split('/')[0]
        Sid = path[1].split('/')[-1]
        if len(path) == 3:
            ExSid = ''
            Subid = path[2].split('.')[0]
        elif len(path) == 4:
            ExSid = '_' + path[2]
            Subid = path[3].split('.')[0]
        else:
            ExSid = '_' + path[2] + '_' + path[3]
            Subid = path[4].split('.')[0]
        if cohort == 'healthy':
            sid_heal_df.iloc[i_heal, :4] = [FCid, Sid, Subid, ExSid]
            i_heal += 1
        else:
            sid_canc_df.iloc[i_canc, :4] = [FCid, Sid, Subid, ExSid]
            i_canc += 1
    return sid_heal_df, sid_canc_df
#
# function to update performance df with predictions
def update_df_pred(df, model, X_test):
    prob_pred = model(X_test, training=False)
    y_pred = np.int32(prob_pred > 0.5)[:, 0]
    df['Pred'] = y_pred
    return df
#
# function for test data accuracy
def testAccSpecSensAUCaucPR(X_test_neg, X_test_pos, Y_test_neg, Y_test_pos):
    #
    test_accuracy = tf.keras.metrics.BinaryAccuracy()
    test_accuracy.update_state(Y_test_neg, model(X_test_neg, training=False))
    test_spec = test_accuracy.result() * 100
    test_accuracy = tf.keras.metrics.BinaryAccuracy()
    test_accuracy.update_state(Y_test_pos, model(X_test_pos, training=False))
    test_sens = test_accuracy.result() * 100
    test_acc = test_spec * len(Y_test_neg) / (len(Y_test_neg) + len(Y_test_pos)) +\
               test_sens * len(Y_test_pos) / (len(Y_test_neg) + len(Y_test_pos))
    test_AUC = tf.keras.metrics.AUC() # ROC AUC
    test_AUC.update_state(np.concatenate((Y_test_neg, Y_test_pos)),\
                          model(np.concatenate((X_test_neg, X_test_pos)), training=False))
    test_AUCPR = tf.keras.metrics.AUC(curve='PR', name='pr_auc') # PR AUC
    test_AUCPR.update_state(np.concatenate((Y_test_neg, Y_test_pos)),\
                          model(np.concatenate((X_test_neg, X_test_pos)), training=False))
    return np.round(test_acc, 3), np.round(test_spec, 3), np.round(test_sens, 3), np.round(test_AUC.result(), 5),\
           np.round(test_AUCPR.result(), 5)
#
def testAccSpecSensAUCaucPR_indivSamples(sid_heal_df, sid_canc_df):
    nSer = sid_heal_df['Sid'].value_counts() # ground truth negatives
    N = len(nSer)
    fpSer = sid_heal_df.loc[sid_heal_df['Pred'] == 1, 'Sid'].value_counts() # candidate false positives
    FP = 0
    # loop through fp candidates and select only those that exceed 50% of their subsamples number
    for candFP in fpSer.index:
        if fpSer[candFP] / nSer[candFP] > 0.5:
            FP += 1
    # loop through all gr.truth negatives and mark FP those that exceed 50% of their subsamples number
    indivPred = []
    for negSID in nSer.index:
        if negSID in fpSer.index:
            if fpSer[negSID] / nSer[negSID] > 0.5:
                indivPred.append(1)
            else:
                indivPred.append(0)
        else:
            indivPred.append(0)
    #
    pSer = sid_canc_df['Sid'].value_counts() # ground truth positives
    P = len(pSer)
    fnSer = sid_canc_df.loc[sid_canc_df['Pred'] == 0, 'Sid'].value_counts() # candidate false negatives
    FN = 0
    # loop through fn candidates and select only those that exceed 50% of their subsamples
    for candFN in fnSer.index:
        if fnSer[candFN] / pSer[candFN] > 0.5:
            FN += 1
    # loop through all gr.truth positives and mark FN those that exceed 50% of their subsamples number
    for posSID in pSer.index:
        if posSID in fnSer.index:
            if fnSer[posSID] / pSer[posSID] > 0.5:
                indivPred.append(0)
            else:
                indivPred.append(1)
        else:
            indivPred.append(1)
    #
    indivLabl = np.concatenate((np.zeros(N, dtype=int), np.ones(P, dtype=int)))
    #
    Sp = (N - FP) / N * 100 # Specificity
    #
    Sn = (P - FN) / P * 100 # Sensitivity
    #
    Ac = (N - FP + P - FN) / (N + P) * 100 # Accuracy
    #
    AUC = tf.keras.metrics.AUC()
    AUC.update_state(indivLabl, np.array(indivPred)) # ROC AUC
    #
    AUCPR = tf.keras.metrics.AUC(curve='PR', name='pr_auc')
    AUCPR.update_state(indivLabl, np.array(indivPred)) # PR AUC
    #
    return np.round(Ac, 3), np.round(Sp, 3), np.round(Sn, 3), np.round(AUC.result(), 5), np.round(AUCPR.result(), 5)
#
def testAccSpecSensAUCaucPR_indivSamples_cons(sid_heal_df, sid_canc_df):
    nSer = sid_heal_df['Sid'].value_counts() # ground truth negatives
    N = len(nSer) # number of ground truth negatives
    fpSer = sid_heal_df.loc[sid_heal_df['Pred'] == 1, 'Sid'].value_counts() # candidate false positives
    FP = len(fpSer) # number of false positives
    # loop through all gr.truth negatives and mark FP those that have at least one such subsample
    indivPred = []
    for negSID in nSer.index:
        if negSID in fpSer.index:
            indivPred.append(1)
        else:
            indivPred.append(0)
    #
    pSer = sid_canc_df['Sid'].value_counts() # ground truth positives
    P = len(pSer) # number of ground truth positives
    fnSer = sid_canc_df.loc[sid_canc_df['Pred'] == 0, 'Sid'].value_counts() # candidate false negatives
    FN = len(fnSer) # number of false negatives
    # loop through all gr.truth positives and mark FN those that have at least one such subsample
    for posSID in pSer.index:
        if posSID in fnSer.index:
            indivPred.append(0)
        else:
            indivPred.append(1)
    #
    indivLabl = np.concatenate((np.zeros(N, dtype=int), np.ones(P, dtype=int)))    
    #
    Sn = (P - FN) / P * 100 # Sensitivity
    Sp = (N - FP) / N * 100 # Specificity
    Ac = (N - FP + P - FN) / (N + P) * 100 # Accuracy
    #
    AUC = tf.keras.metrics.AUC()
    AUC.update_state(indivLabl, np.array(indivPred)) # ROC AUC
    #
    AUCPR = tf.keras.metrics.AUC(curve='PR', name='pr_auc')
    AUCPR.update_state(indivLabl, np.array(indivPred)) # PR AUC
    #
    return np.round(Ac, 3), np.round(Sp, 3), np.round(Sn, 3), np.round(AUC.result(), 5), np.round(AUCPR.result(), 5)
#
#
##
# Training and testing in a loop over different train/test splits and with different initializations of DNN
#
time000 = time.time()
for v in range(1, (Nsplits + 1)):
    #
    # set random seed to match DWT_DNN case
    #tf.random.set_seed(3)
    #
    # load reduced subsampling train/test set paths and load FS pdf profiles in the 51 - 250 bp range
    vers = str(v).zfill(2)
    with open(refDir + 'FSpdfPaths_TrainTest_N4P5_' + vers + '.pickle', 'rb') as handle:
        fpathsTrn, labelsTrn, fpathsTst, labelsTst, splitNumbs = pickle.load(handle)
    #
    # form training and testing datasets of decimated DWT coeffs in 1D
    #X = dec_dwt1D(load_data(fpathsTrn))
    #X_test = dec_dwt1D(load_data(fpathsTst))
    X = load_data(fpathsTrn)
    X_test = load_data(fpathsTst)    
    # optionally, scale data
    #min_max_scaler = MinMaxScaler()
    #X = min_max_scaler.fit_transform(X)
    #X_test = min_max_scaler.transform(X_test)
    #X_test = min_max_scaler.fit_transform(X_test)
    # labels
    Y = np.array(labelsTrn)
    Y_test = np.array(labelsTst)
    #
    # split X(Y)_test in neg and pos parts and reshape
    X_test_neg, X_test_pos, Y_test_neg, Y_test_pos = split_Test_neg_pos(X_test, Y_test)
    #
    # trim, reshape, normalize, shuffle and batch the train dataset
    BUFFER_SIZE = X.shape[0] - (X.shape[0] % BATCH_SIZE)
    X = X[:BUFFER_SIZE, :]
    Y = Y[:BUFFER_SIZE]
    X = X.reshape(BUFFER_SIZE, 1, X.shape[1])
    X = tf.data.Dataset.from_tensor_slices(X).shuffle(BUFFER_SIZE, seed=BUFFER_SIZE).batch(BATCH_SIZE)
    Y = tf.data.Dataset.from_tensor_slices(Y).shuffle(BUFFER_SIZE, seed=BUFFER_SIZE).batch(BATCH_SIZE)
    #
    # performance df
    perf_df = pd.DataFrame(np.empty((Ntr_inst, 20)), index=np.arange(Ntr_inst),\
              columns=['AccSub', 'SpcSub', 'SnsSub', 'AucSub', 'AucPRSub',\
                       'AccMaj', 'SpcMaj', 'SnsMaj', 'AucMaj', 'AucPRMaj',\
                       'AccCon', 'SpcCon', 'SnsCon', 'AucCon', 'AucPRCon',\
                       'LossMean', 'LossRange', 'TrAccMean', 'TrEpochs', 'TrTime'])
    # test dfs
    sid_heal_df, sid_canc_df = make_test_dfs(fpathsTst, labelsTst)
    #
    # train a model with same hyper-parameters several times to assess variability due to randomness in the model behavior
    time00 = time.time()
    #for itr in range(1):
    for itr in range(Ntr_inst):
        # define DNN
        model, cross_entropy, optimizer = DNN_model(out_dim, drop_rate, learning_rate)
        # function for train/loss; needs to be defined for each new model run
        @tf.function
        def train_step(x, y):
            #
            with tf.GradientTape() as tape:
                y_hat = model(x, training=True)
                loss = cross_entropy(y, y_hat)
            #
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            #
            return loss
        #
        time0 = time.time()
        train_loss = []
        train_accuracy = []
        # loop over training epochs
        for epoch in range(EPOCHS):
            epoch_loss_avg = tf.keras.metrics.Mean()
            epoch_accuracy = tf.keras.metrics.BinaryAccuracy()
            # Training loop - using batches of BATCH_SIZE
            for x, y in zip(X, Y):
                loss = train_step(x, y) # Optimize the model
                epoch_loss_avg.update_state(loss) # track batch loss
                epoch_accuracy.update_state(y, model(x, training=True)) # track batch accuracy
            # End epoch
            train_loss.append(np.round(epoch_loss_avg.result(), 3))
            train_accuracy.append(np.round(epoch_accuracy.result() * 100, 3))
            # Terminate training if the conditions are met
            if (epoch >= 4):
                lossMean = np.mean(train_loss[-5:])
                lossRange = np.max(train_loss[-5:]) - np.min(train_loss[-5:])
                TrAccMean = np.round(np.mean(train_accuracy[-5:]), 3)
                if (lossMean <= lossThresh) & (lossRange / lossMean <= rangeThresh):
                    break
        trTime = np.round(time.time() - time0, 2)
        # update dfs with predictions
        sid_heal_df = update_df_pred(sid_heal_df, model, X_test_neg)
        sid_canc_df = update_df_pred(sid_canc_df, model, X_test_pos)    
        # record test metrics
        perf_df.iloc[itr, 0:5] = testAccSpecSensAUCaucPR(X_test_neg, X_test_pos, Y_test_neg, Y_test_pos) # subsample performance
        perf_df.iloc[itr, 5:10] = testAccSpecSensAUCaucPR_indivSamples(sid_heal_df, sid_canc_df) # ind.samp perf, maj-of-vote
        perf_df.iloc[itr, 10:15] = testAccSpecSensAUCaucPR_indivSamples_cons(sid_heal_df, sid_canc_df) # ind.samp perf,consensus
        perf_df.iloc[itr, 15:] = (lossMean, lossRange, TrAccMean, epoch + 1, trTime)
        #
        if itr % 5 == 0:
            print("Tr.inst.: {} Tr.epochs: {} Mean Loss: {:.3f} Loss Range: {:.3f}  Mean Acc: {:.2f}%  Tr.Time: {} sec".format(\
                                                itr+1, epoch+1, lossMean, lossRange, np.round(TrAccMean, 3), trTime))
    #
    print('Split {}:  {} training instances completed in {} min'.format(v, Ntr_inst, np.round((time.time() - time00) / 60, 2)))
    print('   ')
    #
    # save perf_df for each data split on the go
    with open(resDir + fResName + vers + '.pickle', 'wb') as handle:
        pickle.dump(perf_df, handle, protocol=pickle.HIGHEST_PROTOCOL)    
#
print('All {} completed in {} hrs'.format(Nsplits, np.round((time.time() - time000) / 3600, 3)))

Tr.inst.: 1 Tr.epochs: 339 Mean Loss: 0.092 Loss Range: 0.012  Mean Acc: 97.31%  Tr.Time: 33.94 sec
Tr.inst.: 6 Tr.epochs: 274 Mean Loss: 0.098 Loss Range: 0.014  Mean Acc: 97.03%  Tr.Time: 27.84 sec
Tr.inst.: 11 Tr.epochs: 342 Mean Loss: 0.092 Loss Range: 0.012  Mean Acc: 97.27%  Tr.Time: 33.48 sec
Tr.inst.: 16 Tr.epochs: 304 Mean Loss: 0.100 Loss Range: 0.011  Mean Acc: 97.07%  Tr.Time: 32.73 sec
Split 1:  20 training instances completed in 10.53 min
   
Tr.inst.: 1 Tr.epochs: 351 Mean Loss: 0.099 Loss Range: 0.013  Mean Acc: 96.99%  Tr.Time: 35.28 sec
Tr.inst.: 6 Tr.epochs: 373 Mean Loss: 0.094 Loss Range: 0.013  Mean Acc: 97.50%  Tr.Time: 38.47 sec
Tr.inst.: 11 Tr.epochs: 400 Mean Loss: 0.104 Loss Range: 0.054  Mean Acc: 97.07%  Tr.Time: 39.83 sec
Tr.inst.: 16 Tr.epochs: 400 Mean Loss: 0.095 Loss Range: 0.021  Mean Acc: 97.38%  Tr.Time: 39.85 sec
Split 2:  20 training instances completed in 12.77 min
   
Tr.inst.: 1 Tr.epochs: 400 Mean Loss: 0.105 Loss Range: 0.044  Mean Acc: 97.03